In [1]:
!pip install -q transformers tokenizers

In [2]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import json
import os

from collections import defaultdict

import torch
import transformers
import re

from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          AutoModelForMultipleChoice,
                          pipeline)

Reference: https://colab.research.google.com/github/huggingface/notebooks/blob/main/examples/multiple_choice.ipynb

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
model_name = "bert-base-uncased"
data_dir = "/content/drive/MyDrive/Master's/Second Year Grad/NLU/NLU_FinalProject/Data/JSONL_Formatted/"

data_path = "RACE-H/RACE-H_test.jsonl"
data_name = 'RACE-H_test'
save_dir = './'

Load Model & Tokenizer

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Load and Process Data

In [ ]:
df = pd.read_json(data_dir + data_path, lines=True)
df.head()

,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high17205.txt,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",D,people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...
1,high17205.txt,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",C,the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science
2,high17205.txt,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,A,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians
3,high17226.txt,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,B,what a website is like,how to build your own website,how to meet people online,what a website is made up of
4,high17226.txt,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",D,where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet


In [ ]:
#ans_names = ["answerA", "answerB", "answerC", "answerD"]
ans_names = ["mc_a", "mc_b", "mc_c", "mc_d"]

def preprocess_function(row):
    # Repeat each question four times to go with the four possibilities of second sentences.
    first_sentences = [row.prompt + row.question] * 4
    # Grab all answers possible for each question.
    second_sentences = [row[name] for name in ans_names]

    # Tokenize
    tokenized_examples = tokenizer(first_sentences, second_sentences, padding=True, truncation=True, return_tensors='pt').to(device)
    return {k: v.view(1, 4, -1) for k, v in tokenized_examples.items()}

model_input = df.apply(preprocess_function, axis=1)
print(model_input.iloc[0])
print(len(model_input.iloc[0]["input_ids"]), [len(x) for x in model_input.iloc[0]['input_ids']])
print(f'Number of Questions = {len(model_input):,d}')

{'input_ids': tensor([[[  101,  2009,  2003,  ..., 13966,   102,     0],
         [  101,  2009,  2003,  ...,  3404, 13966,   102],
         [  101,  2009,  2003,  ...,   102,     0,     0],
         [  101,  2009,  2003,  ...,   102,     0,     0]]], device='cuda:0'), 'token_type_ids': tensor([[[0, 0, 0,  ..., 1, 1, 0],
         [0, 0, 0,  ..., 1, 1, 1],
         [0, 0, 0,  ..., 1, 0, 0],
         [0, 0, 0,  ..., 1, 0, 0]]], device='cuda:0'), 'attention_mask': tensor([[[1, 1, 1,  ..., 1, 1, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 0, 0],
         [1, 1, 1,  ..., 1, 0, 0]]], device='cuda:0')}
1 [4]
Number of Questions = 3,498


In [ ]:
[tokenizer.decode(model_input.iloc[0]["input_ids"][0][i]) for i in range(4)]

['[CLS] it is the goal of politicians everywhere - - - - - how to win and keep the trust of voters. now researchers at the university of st anurew \' s in scotland say they may have the answer. they believe politicians could learn a lot from recent advances in science. a growing number of studies have shown that people do judge a book by its cover. researchers say most of us make quick judgments about a person on the basis of how they look. studies suggest that people are less likely to trust those with particularly masculine features, such as a square jaw, small eyes or a big nose. " they are considered dominant and less trustworthy, " says ms cornwell. " it doesn \' t mean that men who look more masculine are less trustworthy - - - - - it \' s just our first impression. " those with less masculine features - - - - - larger eyes, a smaller nose and thinner lips are thought to be more trustworthy. the researchers are putting their science to the test at the royal society \' s annual su

Run Inference

In [ ]:
# test output
example_question = model_input.iloc[0]
output = model(**example_question)

In [ ]:
torch.softmax(output.logits[0], dim=-1)

tensor([0.2503, 0.2507, 0.2480, 0.2509], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [ ]:
pred = 'ABCD'[torch.argmax(torch.softmax(output.logits[0], dim=-1), dim=-1).item()]
pred

'D'

In [ ]:
res = defaultdict(list)

for text in tqdm(model_input):
  try:
    output = model(**text)
    probs = torch.softmax(output.logits[0], dim=-1).to('cpu')
  except:
    probs = None

  res['prob_A'].append(probs[0].item() if probs is not None else None)
  res['prob_B'].append(probs[1].item() if probs is not None else None)
  res['prob_C'].append(probs[2].item() if probs is not None else None)
  res['prob_D'].append(probs[3].item() if probs is not None else None)
  res['pred'].append('ABCD'[torch.argmax(probs, dim=-1).item()] if probs is not None else None)

100%|██████████| 3498/3498 [05:59<00:00,  9.74it/s]


In [ ]:
res_df = pd.concat([pd.DataFrame(res), df], axis=1)
res_df.head()

,prob_A,prob_B,prob_C,prob_D,pred,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,0.250340,0.250749,0.247971,0.250939,D,high17205.txt,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",D,people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...
1,0.250819,0.249451,0.250832,0.248899,C,high17205.txt,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",C,the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science
2,0.245127,0.242511,0.258370,0.253993,C,high17205.txt,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,A,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians
3,0.241820,0.240743,0.274257,0.243179,C,high17226.txt,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,B,what a website is like,how to build your own website,how to meet people online,what a website is made up of
4,0.266533,0.283635,0.216414,0.233418,B,high17226.txt,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",D,where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet


In [ ]:
print(res_df.pred.value_counts())

pred
A    924
D    869
B    854
C    851
Name: count, dtype: int64


In [ ]:
# evaluation metrics

# accuracy
accuracy = sum(res_df.pred == res_df.answer)/len(res_df.dropna())

# percent response failure
res_fail = sum(res_df.pred.isnull())/len(res_df)

df_eval = pd.DataFrame({'dataset':[data_name],
                        'accuracy':[accuracy],
                        'fail_rate':[res_fail],})

if os.path.exists(f"{save_dir}benchmark_summary_{model_name}.csv"):
  df_temp = pd.read_csv(f"{save_dir}benchmark_summary_{model_name}.csv")
  df_eval = pd.concat([df_eval, df_temp], axis=0, ignore_index=True)

df_eval

,dataset,accuracy,fail_rate
0,RACE-H_test,0.215266,0.0
1,SAT-ACT_test,0.242424,0.0


Store Results

In [ ]:
res_df.to_csv(f"{save_dir}{data_name}_benchmark_{model_name}.csv", index=False)

In [ ]:
df_eval.to_csv(f"{save_dir}benchmark_summary_{model_name}.csv", index=False)